# Merge datasets

> merge multiple datasets(train and test(validation)) and their annotations

In [ ]:
#| default_exp preprocessing.merge_pin_datasets_multi

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import json
import os
import shutil

## `merge_datasets_multi` — Build from scratch

Takes a **list of datasets** and builds a merged COCO directory from the ground up.  
All image IDs and annotation IDs start from 1. Use this when creating a new merged dataset for the first time.

In [ ]:
#| export
def merge_datasets_multi(datasets: list, output_path: str):
    """
    Merge multiple COCO datasets into one multi-class dataset.
    
    Args:
        datasets: List of dicts with 'path', 'category_name', and 'prefix'
                  Example: [
                      {"path": "/path/to/pins", "category_name": "pin", "prefix": "a"},
                      {"path": "/path/to/eberhard", "category_name": "eberhard_pin", "prefix": "b"},
                      {"path": "/path/to/new_type", "category_name": "new_pin", "prefix": "c"},
                  ]
        output_path: Where to save merged dataset
    """
    
    for split in ["train", "test"]:
        
        # Create output directories
        os.makedirs(f"{output_path}/{split}/images", exist_ok=True)
        
        # Dynamically create categories from input
        categories = [
            {"id": i + 1, "name": ds["category_name"]} 
            for i, ds in enumerate(datasets)
        ]
        
        # Add info section before images
        info = {
            "description": "3_pin_1_crack Dataset",
            "version": "1.0",
            "year": 2025
        }
        
        # Initialize merged structure
        merged = {
            "info": info,
            "images": [],
            "categories": categories,
            "annotations": []
        }
        
        img_id_counter = 1
        ann_id_counter = 1
        
        # Process each dataset
        for ds_idx, ds_config in enumerate(datasets):
            ds_path = ds_config["path"]
            prefix = ds_config["prefix"]
            category_id = ds_idx + 1  # 1-indexed
            
            # Load annotation file
            ann_path = f"{ds_path}/{split}/annotation.json"
            if not os.path.exists(ann_path):
                print(f"  Skipping {split} - {ds_config['category_name']}: no {split} folder found")
                continue
            with open(ann_path, "r") as f:
                data = json.load(f)
            
            # Map old image IDs to new IDs
            old_to_new_img = {}
            
            # Process images
            for img in data["images"]:
                old_id = img["id"]
                new_img = {
                    "id": img_id_counter,
                    "file_name": f"{prefix}_{img['file_name']}",
                    "width": img["width"],
                    "height": img["height"]
                }
                merged["images"].append(new_img)
                old_to_new_img[old_id] = img_id_counter
                img_id_counter += 1
                
                # Copy image
                src = f"{ds_path}/{split}/images/{img['file_name']}"
                dst = f"{output_path}/{split}/images/{prefix}_{img['file_name']}"
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                else:
                    print(f"  Warning: Missing {src}")
            
            # Process annotations
            for ann in data["annotations"]:
                new_ann = {
                    "id": ann_id_counter,
                    "image_id": old_to_new_img[ann["image_id"]],
                    "category_id": category_id,  # Assign correct category
                    "bbox": ann["bbox"],
                    "segmentation": ann.get("segmentation", []),
                    "area": ann["area"],
                    "iscrowd": ann.get("iscrowd", 0)
                }
                merged["annotations"].append(new_ann)
                ann_id_counter += 1
            
            print(f"  {split} - {ds_config['category_name']}: {len(data['images'])} images")
        
        # Save merged annotation
        output_ann_path = f"{output_path}/{split}/annotation.json"
        with open(output_ann_path, "w") as f:
            json.dump(merged, f, indent=2)
        
        print(f"  {split} TOTAL: {len(merged['images'])} images, {len(merged['annotations'])} annotations")
        print(f"  Categories: {[c['name'] for c in categories]}")


In [ ]:

#if __name__ == "__main__":
#    TYPE_1_PATH = "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/pins"    
#    TYPE_2_PATH = "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/eberhard_pins"      
#    OUTPUT_PATH = "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/combined_pins"   
#    
#    merge_datasets(TYPE_1_PATH, TYPE_2_PATH, OUTPUT_PATH)

In [ ]:
#| export
if __name__ == "__main__":
    datasets = [
        {
            "path": "/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/with_hn" ,
            "category_name": "et_pin",
            "prefix": "a"
        },
        {
            "path": "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/eberhard_pins",
            "category_name": "eberhard_pin",
            "prefix": "b"
        },
        {
            "path": "/home/hpc_storage/data/projects/userA/sam3_data/wuxi_pin/with_hn" ,
            "category_name": "wuxi_pin",
            "prefix": "c"
        },
        {
            "path": "/home/hpc_storage/data/projects/userA/sam3_data/jubna_images",
            "category_name": "war_crack",
            "prefix": "d"
        },
    ]
    
    OUTPUT_PATH = "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/3pin_1crack_6"
    
    merge_datasets_multi(datasets, OUTPUT_PATH)

## `append_dataset` — Add a category to an existing merged directory

Instead of rebuilding from scratch, this reads the **existing** `annotation.json`, finds the max IDs already in use, and appends only the new category's images and annotations.  
The existing data is untouched — only the new entries are added and the file is overwritten with the combined result.

**Scenario below:** the merged directory already has 3 categories (`et_pin`, `eberhard_pin`, `wuxi_pin`) and we are adding `war_crack` as the 4th.

In [ ]:
#| export
def append_dataset(new_dataset: dict, output_path: str):
    """
    Append a single new category to an existing merged COCO directory.

    Args:
        new_dataset: Dict with 'path', 'category_name', and 'prefix'
        output_path: Path to the existing merged dataset directory
    """
    for split in ["train", "test"]:
        ann_path = f"{output_path}/{split}/annotation.json"
        if not os.path.exists(ann_path):
            print(f"  Skipping {split}: no existing annotation.json found")
            continue

        # Load existing merged annotation
        with open(ann_path, "r") as f:
            merged = json.load(f)

        # Continue IDs from where the existing data left off
        img_id_counter  = max((img["id"] for img in merged["images"]),      default=0) + 1
        ann_id_counter  = max((ann["id"] for ann in merged["annotations"]), default=0) + 1
        category_id     = max((c["id"]   for c   in merged["categories"]),  default=0) + 1

        merged["categories"].append({"id": category_id, "name": new_dataset["category_name"]})

        # Load new dataset annotation
        new_ann_path = f"{new_dataset['path']}/{split}/annotation.json"
        if not os.path.exists(new_ann_path):
            print(f"  Skipping {split} - {new_dataset['category_name']}: no annotation.json found")
            continue

        with open(new_ann_path, "r") as f:
            data = json.load(f)

        prefix = new_dataset["prefix"]
        old_to_new_img = {}

        for img in data["images"]:
            old_id = img["id"]
            merged["images"].append({
                "id": img_id_counter,
                "file_name": f"{prefix}_{img['file_name']}",
                "width": img["width"],
                "height": img["height"]
            })
            old_to_new_img[old_id] = img_id_counter
            img_id_counter += 1

            src = f"{new_dataset['path']}/{split}/images/{img['file_name']}"
            dst = f"{output_path}/{split}/images/{prefix}_{img['file_name']}"
            if os.path.exists(src):
                shutil.copy2(src, dst)
            else:
                print(f"  Warning: Missing {src}")

        for ann in data["annotations"]:
            merged["annotations"].append({
                "id": ann_id_counter,
                "image_id": old_to_new_img[ann["image_id"]],
                "category_id": category_id,
                "bbox": ann["bbox"],
                "segmentation": ann.get("segmentation", []),
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
            ann_id_counter += 1

        with open(ann_path, "w") as f:
            json.dump(merged, f, indent=2)

        print(f"  {split} - {new_dataset['category_name']}: {len(data['images'])} images added")
        print(f"  {split} TOTAL: {len(merged['images'])} images, {len(merged['annotations'])} annotations")
        print(f"  Categories: {[c['name'] for c in merged['categories']]}")


In [ ]:
# Existing merged dir already has: et_pin, eberhard_pin, wuxi_pin
# Appending war_crack as the 4th category

if __name__ == "__main__":
    EXISTING_OUTPUT = "/home/hpc_storage/data/projects/userA/sam3_data/multi_dataset/2_cat_pins_2"

    wuxi_pin = {
            "path": "/home/hpc_storage/data/projects/userA/sam3_data/jubna_images",
            "category_name": "war_crack",
            "prefix": "d"
    }

    append_dataset(wuxi_pin, EXISTING_OUTPUT)


## `append_images_to_existing_category` — Add more images to a category that already exists

`append_dataset()` above always creates a **brand-new** `category_id` for `new_dataset["category_name"]`,
even if a category with that exact name already exists in the merged directory. That's correct when
adding a genuinely new class, but wrong when you just collected **more images for a category you
already merged in before** (e.g. more `matrix_chip` samples) — re-running `append_dataset()` in that
case would silently create a *second*, duplicate `matrix_chip` category with a different id, splitting
what should be one class into two.

This function is the counterpart for that case:

- It looks up `new_dataset["category_name"]` in the existing merged `categories` list and **reuses**
  that `category_id` instead of minting a new one.
- It does **not** append a new entry to `categories` — the category list length is unchanged.
- If the category name is **not** found, it raises a `ValueError` rather than silently creating one
  (that would just be `append_dataset()`'s job — use that instead for a genuinely new category).

Image/annotation id remapping, filename prefixing, and file copying are otherwise identical to
`append_dataset()`.


In [ ]:
#| export
def append_images_to_existing_category(new_dataset: dict, output_path: str):
    """
    Add more images to a category that already exists in a merged COCO directory.

    Unlike `append_dataset`, this reuses the existing `category_id` for
    `new_dataset["category_name"]` instead of creating a new one, and raises if
    that category name isn't found (use `append_dataset` for a genuinely new category).

    Args:
        new_dataset: Dict with 'path', 'category_name', and 'prefix'
        output_path: Path to the existing merged dataset directory
    """
    for split in ["train", "test"]:
        ann_path = f"{output_path}/{split}/annotation.json"
        if not os.path.exists(ann_path):
            print(f"  Skipping {split}: no existing annotation.json found")
            continue

        # Load existing merged annotation
        with open(ann_path, "r") as f:
            merged = json.load(f)

        # Find the existing category by name - reuse its id, don't mint a new one
        existing_category = next(
            (c for c in merged["categories"] if c["name"] == new_dataset["category_name"]),
            None,
        )
        if existing_category is None:
            raise ValueError(
                f"Category '{new_dataset['category_name']}' not found in {ann_path}'s "
                f"categories ({[c['name'] for c in merged['categories']]}). "
                "Use append_dataset() to add it as a new category instead."
            )
        category_id = existing_category["id"]

        # Continue IDs from where the existing data left off
        img_id_counter = max((img["id"] for img in merged["images"]), default=0) + 1
        ann_id_counter = max((ann["id"] for ann in merged["annotations"]), default=0) + 1

        # Load new dataset annotation
        new_ann_path = f"{new_dataset['path']}/{split}/annotation.json"
        if not os.path.exists(new_ann_path):
            print(f"  Skipping {split} - {new_dataset['category_name']}: no annotation.json found")
            continue

        with open(new_ann_path, "r") as f:
            data = json.load(f)

        prefix = new_dataset["prefix"]
        old_to_new_img = {}

        for img in data["images"]:
            old_id = img["id"]
            merged["images"].append({
                "id": img_id_counter,
                "file_name": f"{prefix}_{img['file_name']}",
                "width": img["width"],
                "height": img["height"]
            })
            old_to_new_img[old_id] = img_id_counter
            img_id_counter += 1

            src = f"{new_dataset['path']}/{split}/images/{img['file_name']}"
            dst = f"{output_path}/{split}/images/{prefix}_{img['file_name']}"
            if os.path.exists(src):
                shutil.copy2(src, dst)
            else:
                print(f"  Warning: Missing {src}")

        for ann in data["annotations"]:
            merged["annotations"].append({
                "id": ann_id_counter,
                "image_id": old_to_new_img[ann["image_id"]],
                "category_id": category_id,
                "bbox": ann["bbox"],
                "segmentation": ann.get("segmentation", []),
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
            ann_id_counter += 1

        with open(ann_path, "w") as f:
            json.dump(merged, f, indent=2)

        print(f"  {split} - {new_dataset['category_name']} (category_id={category_id}): {len(data['images'])} images added")
        print(f"  {split} TOTAL: {len(merged['images'])} images, {len(merged['annotations'])} annotations")
        print(f"  Categories: {[c['name'] for c in merged['categories']]}")


### Test: `append_images_to_existing_category`

Verifies two things, entirely inside a `tempfile.TemporaryDirectory()` (no real dataset touched):

1. **Happy path** — a merged dataset already has a `matrix_chip` category (id `2`, alongside `et_pin`
   id `1`). Appending a new batch of `matrix_chip` images reuses id `2`: the `categories` list length
   stays the same (2), the `matrix_chip` image count increases by the new batch size, and the newly
   added annotations carry `category_id == 2`.
2. **Edge case** — appending a category name that doesn't exist in the merged dataset (`unknown_cat`)
   raises `ValueError` instead of silently creating a third category.


In [ ]:
import tempfile
from pathlib import Path
from PIL import Image


def _make_test_image(path, size=(4, 4), color=0):
    Image.new("L", size, color=color).save(path)


with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    # --- Build a tiny existing merged dataset: et_pin (id 1), matrix_chip (id 2) ---
    merged_root = tmp / "merged"
    for split in ["train", "test"]:
        (merged_root / split / "images").mkdir(parents=True)
        _make_test_image(merged_root / split / "images" / "a_img1.png")
        _make_test_image(merged_root / split / "images" / "e_img1.png")
        ann = {
            "images": [
                {"id": 1, "file_name": "a_img1.png", "width": 4, "height": 4},
                {"id": 2, "file_name": "e_img1.png", "width": 4, "height": 4},
            ],
            "annotations": [
                {"id": 1, "image_id": 1, "category_id": 1, "bbox": [0, 0, 2, 2], "segmentation": [], "area": 4, "iscrowd": 0},
                {"id": 2, "image_id": 2, "category_id": 2, "bbox": [0, 0, 2, 2], "segmentation": [], "area": 4, "iscrowd": 0},
            ],
            "categories": [{"id": 1, "name": "et_pin"}, {"id": 2, "name": "matrix_chip"}],
        }
        with open(merged_root / split / "annotation.json", "w") as f:
            json.dump(ann, f)

    # --- Build a small "new batch" of matrix_chip images to append ---
    new_root = tmp / "new_matrix_chip_batch"
    for split in ["train", "test"]:
        (new_root / split / "images").mkdir(parents=True)
        _make_test_image(new_root / split / "images" / "imgA.png")
        ann = {
            "images": [{"id": 1, "file_name": "imgA.png", "width": 4, "height": 4}],
            "annotations": [
                {"id": 1, "image_id": 1, "category_id": 1, "bbox": [0, 0, 2, 2], "segmentation": [], "area": 4, "iscrowd": 0}
            ],
            "categories": [{"id": 1, "name": "matrix_chip"}],
        }
        with open(new_root / split / "annotation.json", "w") as f:
            json.dump(ann, f)

    new_dataset = {"path": str(new_root), "category_name": "matrix_chip", "prefix": "f"}

    # --- Happy path ---
    append_images_to_existing_category(new_dataset, str(merged_root))

    with open(merged_root / "train" / "annotation.json") as f:
        result = json.load(f)

    n_categories_ok = len(result["categories"]) == 2
    n_images_ok = len(result["images"]) == 3  # 2 original + 1 new
    new_ann = next(a for a in result["annotations"] if a["image_id"] == 3)
    category_id_reused_ok = new_ann["category_id"] == 2  # reused matrix_chip's id, not a new one

    print("✓ category count unchanged (no duplicate)" if n_categories_ok else f"✗ expected 2 categories, got {len(result['categories'])}")
    print("✓ image count increased by new batch size" if n_images_ok else f"✗ expected 3 images, got {len(result['images'])}")
    print("✓ new annotation reuses existing category_id" if category_id_reused_ok else f"✗ expected category_id=2, got {new_ann['category_id']}")

    # --- Edge case: unknown category name must raise, not silently create one ---
    unknown_dataset = {"path": str(new_root), "category_name": "unknown_cat", "prefix": "z"}
    try:
        append_images_to_existing_category(unknown_dataset, str(merged_root))
        print("✗ expected ValueError for unknown category name, none raised")
    except ValueError as e:
        print(f"✓ unknown category name correctly raised ValueError: {e}")


In [ ]:
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_fine_tune/nbs'
import os
os.chdir(CURRENT_NB)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()